# Notebook 05 — Time-Series Forecasting with SARIMA

**Phase 5 learning checkpoint.** We shift from cross-sectional regression (predicting *one home's* price from its features) to time-series forecasting (predicting *a ZIP code's* median price trajectory into the future).

## What you will do here

1. Reshape the wide Zillow ZHVI CSV into tidy (long) format.
2. Pick a handful of Iowa ZIP codes and plot their raw price history.
3. Decompose one series into trend + seasonal + residual.
4. Fit a SARIMA model on a single ZIP and inspect the fit.
5. Generate a 24-month forecast with 95% confidence intervals.
6. Forecast several ZIPs at once and compare them on a grid.
7. Export the forecasts to CSV for Phase 7 (Power BI).

## Reading order

There is no separate doc for Phase 5 yet — the code comments in `src/forecasting.py` serve as the explainer. Read them alongside each cell.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 120)

from src.data_loader import load_zillow_zhvi
from src import forecasting as F

print('Setup complete.')

## 1. Load and reshape the Zillow data

The raw CSV is **wide**: one row per ZIP, one column per month-end date. That's convenient for storage but hard to work with in Python. `melt_zhvi` reshapes it to **long** (tidy) format: one row per `(zip, month)` pair.

| Format | Shape | Good for |
|--------|-------|----------|
| Wide   | 26k rows × 320+ cols | storage, Excel pivot tables |
| Long   | 8M+ rows × 4 cols | filtering, groupby, time-series models |

In [ ]:
raw = load_zillow_zhvi()
print(f'Raw shape: {raw.shape}')

long = F.melt_zhvi(raw)
print(f'Long shape: {long.shape}')
long.head(10)

## 2. Pick ZIP codes to work with

We use Iowa ZIPs to stay connected to the Ames housing data from earlier phases. `get_top_zips` ranks ZIPs by how many months of data they have — longer series make better SARIMA models.

In [ ]:
iowa_zips = F.get_top_zips(long, n=6, state='IA')
print('Top Iowa ZIPs by data completeness:', iowa_zips)

iowa = F.filter_zips(long, iowa_zips)
print(f'Filtered: {iowa.shape[0]:,} rows, {iowa["zip"].nunique()} ZIPs')
iowa.groupby('zip')[['zhvi']].agg(['count', 'min', 'max', 'mean']).round(0)

## 3. Explore raw price history

Before fitting any model, always **look at the data**. Key questions:
- Is there a clear upward trend?
- Do you see seasonal cycles (prices peak in summer, dip in winter)?
- Are there sudden jumps (COVID-era surge, 2008 crash)?

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

for z in iowa_zips:
    s = iowa[iowa['zip'] == z].set_index('date')['zhvi'].sort_index()
    ax.plot(s.index.to_timestamp(), s.values, linewidth=1.5, label=f'ZIP {z}')

ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_title('Iowa ZIP Codes — ZHVI Monthly History', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('ZHVI ($)')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Seasonal decomposition

Before fitting ARIMA, it helps to decompose a series into its three components:

| Component | Meaning |
|-----------|---------|
| **Trend** | Long-run direction (prices rising over years) |
| **Seasonal** | Repeating pattern within each year (summer peaks) |
| **Residual** | What's left — random noise the model can't explain |

A large seasonal component → include seasonal terms in SARIMA (the `S` part).
A noisy residual → the series has genuine unpredictability; forecast intervals will be wide.

In [ ]:
focus_zip = iowa_zips[0]   # change this to explore a different ZIP
print(f'Decomposing ZIP {focus_zip}')

fig = F.decompose(long, zip_code=focus_zip, model='additive')
plt.show()

## 5. Fit a SARIMA model and inspect it

`forecast_zip` runs an AIC grid search to pick good `(p,d,q)(P,D,Q)` orders, then fits the model. This takes ~30 seconds per ZIP.

**What is AIC?** The Akaike Information Criterion balances goodness-of-fit against complexity. A lower AIC means "this model explains the data almost as well, but with fewer parameters" — the sweet spot between underfitting and overfitting.

In [ ]:
print(f'Fitting SARIMA for ZIP {focus_zip} (auto order selection)...')
fc_single = F.forecast_zip(long, zip_code=focus_zip, periods=24, auto=True)

print(f'\nForecast ({len(fc_single)} months):')
fc_single.head(12)

## 6. Plot the forecast

The **dashed red line** is the point forecast. The **shaded ribbon** is the 95% confidence interval — the range where we expect the true value to fall 95% of the time.

Notice how the ribbon widens as we go further into the future. That's correct: uncertainty compounds over time. A 2-year forecast is inherently less certain than a 3-month one.

In [ ]:
F.plot_forecast(
    focus_zip,
    long_df=long,
    forecast_df=fc_single,
    history_months=60,
)
plt.show()

## 7. Forecast all Iowa ZIPs

We loop over all six ZIPs. `forecast_multiple_zips` handles the loop and prints progress. Expect ~30 seconds per ZIP.

In [ ]:
print('Forecasting all Iowa ZIPs (this takes a couple of minutes)...')
forecasts = F.forecast_multiple_zips(long, iowa_zips, periods=24, auto=True)
print(f'\nDone. Forecasted {len(forecasts)} ZIPs.')

## 8. Compare all ZIPs side by side

The grid below lets you visually compare trends across ZIPs. Look for:
- Which ZIPs are appreciating fastest?
- Which have the widest confidence intervals (most uncertain)?
- Are any ZIPs projected to plateau or decline?

In [ ]:
fig = F.plot_multi_forecast(
    iowa_zips,
    long_df=long,
    forecasts=forecasts,
    history_months=60,
)
plt.show()

## 9. Forecast summary table

Pull the 12-month and 24-month point forecasts for each ZIP into a single comparison table.

In [ ]:
rows = []
for z, fc in forecasts.items():
    current = long[long['zip'] == z]['zhvi'].iloc[-1]
    fc12 = fc.iloc[11]['forecast']
    fc24 = fc.iloc[23]['forecast']
    rows.append({
        'zip': z,
        'current_zhvi': current,
        'forecast_12m': fc12,
        'forecast_24m': fc24,
        'change_12m_%': (fc12 - current) / current * 100,
        'change_24m_%': (fc24 - current) / current * 100,
    })

summary = pd.DataFrame(rows).set_index('zip')
summary[['current_zhvi', 'forecast_12m', 'forecast_24m']] = \
    summary[['current_zhvi', 'forecast_12m', 'forecast_24m']].applymap(lambda x: f'${x:,.0f}')
summary[['change_12m_%', 'change_24m_%']] = \
    summary[['change_12m_%', 'change_24m_%']].applymap(lambda x: f'{x:+.1f}%')
summary

## 10. Save forecasts for Phase 7

Phase 7 will import this CSV into Power BI to build an interactive dashboard. We save it now so it's ready.

In [ ]:
out_path = F.save_forecasts(forecasts, name='iowa_zhvi_forecasts')
print(f'Saved to: {out_path}')

## 11. Wrap-up — what did you learn?

Answer these before moving to Phase 6:

1. **Wide vs. long data.** Why is the tidy (long) format better for modeling even though it has far more rows?

2. **Trend vs. seasonality.** From the decomposition plot, does the Iowa ZIP you examined have strong seasonality? What months tend to be peaks? Does that match intuition about home-buying cycles?

3. **Confidence intervals.** Why do the forecast ribbons widen as you go further out? What would it mean if the ribbon stayed the same width?

4. **ARIMA vs. XGBoost.** In Phase 4 we used XGBoost on Ames data. Why can't we just use XGBoost here too? What does ARIMA capture that XGBoost (applied naively) would miss?

5. **Limitations.** Name two things SARIMA cannot account for that could make its forecasts wrong (think: external shocks, structural breaks, local events).

6. **Phase 7 connection.** The saved CSV has columns `zip`, `date`, `forecast`, `lower_95`, `upper_95`. How would you use all five columns in a Power BI dashboard to communicate both the point estimate *and* the uncertainty?

When you can answer these, you are ready for **Phase 6: Interpretability & ROI**.